# FrameWeave Hybrid v6.0 — TSO abstention mask (Ovis2.5-9B)

Chỉ inference trên 152 câu Temporal Shot Ordering. Không train, không đổi prompt, không scale $\Delta X$. Kết quả ghi ở `outputs_tso_abstention`, không ghi đè CSV residual-direction.

Bốn điều kiện dùng chung `Hybrid.pt` và cùng một vector score cho mỗi model:

| Điều kiện           | Hook                    | Chọn đáp án                |
| ------------------- | ----------------------- | -------------------------- |
| vanilla_unmasked    | bypass                  | argmax trên mọi letter gốc |
| frameweave_unmasked | cộng $\Delta X$ đã chốt | cùng argmax                |
| vanilla_masked      | cùng score Vanilla      | abstention = $-\infty$     |
| frameweave_masked   | cùng score FrameWeave   | abstention = $-\infty$     |

Smoke 8 câu ghi `/content/outputs_tso_abstention/smoke`. Full 152 câu ghi `/content/outputs_tso_abstention/full`.

`--results-path` là ZIP hoặc CSV của HERBench lite eval, hoặc của thí nghiệm residual-direction. ZIP lite lấy `predictions/predictions_reljoin_*.csv`. ZIP residual-direction lấy `predictions_residual_direction/predictions_residual_*.csv`. Bản trong `smoke/` và bản copy Drive không dùng.

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `reljoin/` và `run_*.py` vào `/content`. Không upload file source.

Checkpoint v3/v4/v5 không được load. Có thể tái sử dụng `ovis_env`, Ovis2.5-9B và video lite_v2 của notebook eval.



## 1. Configuration


In [ ]:
# 1. TSO abstention config — HERBench lite_v2 / fixed Hybrid.pt
from __future__ import annotations

from pathlib import Path

assert Path("/content").exists(), "Notebook này chỉ chạy trên Google Colab (/content)."

ROOT = Path("/content")
SMOKE_FIRST = True          # lần đầu True; smoke 8 câu OK rồi đặt False trước full
RUN_FULL = True             # sau smoke: 152 câu, --smoke 0 --resume
SMOKE_N = 8
RESULTS_PATH = Path("")     # để trống: tìm predictions/predictions_reljoin_*.csv hoặc ZIP eval/residual-direction
METHOD_VERSION = "6.0.0"

VARIANT_TO_METHOD = {
    "Hybrid": "hybrid",
}
REQUIRED_CKPTS = ["Hybrid.pt"]

EXTRACT_DIR_CANDIDATES = [
    ROOT / "outputs_reljoin_v6_ovis",
    ROOT / "outputs_reljoin_v6_ovis (2)",
    Path("/content/drive/MyDrive/outputs_reljoin_v6_ovis"),
    Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs") / "outputs_reljoin_v6_ovis",
    Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs"),
]

MODEL_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
HERBENCH_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"

VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"
CKPT_DIR = OUT_DIR / "checkpoints" / "best_external_val"
TSO_ROOT = ROOT / "outputs_tso_abstention"
SMOKE_DIR = TSO_ROOT / "smoke"
FULL_DIR = TSO_ROOT / "full"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
EVAL_DIR = ROOT / "herbench_eval_pins"

print({
    "SMOKE_FIRST": SMOKE_FIRST,
    "RUN_FULL": RUN_FULL,
    "SMOKE_N": SMOKE_N,
    "RESULTS_PATH": str(RESULTS_PATH),
    "SMOKE_DIR": str(SMOKE_DIR),
    "FULL_DIR": str(FULL_DIR),
    "REQUIRED_CKPTS": REQUIRED_CKPTS,
    "CKPT_DIR": str(CKPT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "method": "FrameWeave-Hybrid",
    "method_version": METHOD_VERSION,
    "backbone": MODEL_ID,
})


## 2. Environment (`ovis_env`)

Official Ovis2.5-9B pins: `torch==2.4.0`, `transformers==4.51.3`, `flash-attn==2.7.0.post2`.
Stop if flash-attn install fails — the ViT hard-imports it.


In [ ]:
# 2. Bootstrap isolated ovis_env (Python 3.11 + torch 2.4.0 + flash-attn 2.7)
from __future__ import annotations

import os
import subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "ovis_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs_reljoin_v6_ovis" / "logs" / "cell2_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc


sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "datasets", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging",
])
print("Installing flash-attn==2.7.0.post2 (required by Ovis ViT)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, flash_attn; "
    "print(torch.__version__); print(transformers.__version__); print(flash_attn.__version__); "
    "print(torch.__file__); print(transformers.__file__)",
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/ovis_env/" in line, f"Package leaked outside venv: {line}"
print("ovis_env ready:", PYTHON)
print("LOG:", LOG)


## 3. Drive + pin HERBench evaluators

Không tải lại evaluator nếu SHA đã khớp (tránh GitHub 429). Không clone DTR.


In [ ]:
# 3. Drive + pin HERBench evaluator (skip download if SHA matches)
from __future__ import annotations

import hashlib
import os
import time
import urllib.request
from pathlib import Path

ROOT = Path("/content")
VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"
EVAL_DIR = ROOT / "herbench_eval_pins"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
JSDELIVR_BASE = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/model_wrappers/base_vlm.py"
JSDELIVR_ACC = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/calculate_accuracy.py"

for p in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent, EVAL_DIR, OUT_DIR, ROOT / "outputs_tso_abstention"]:
    p.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    RELJOIN_PERSIST = Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
if RELJOIN_PERSIST.resolve() == OUT_DIR.resolve():
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
    print("Persist dir was out_dir; using", RELJOIN_PERSIST)
RELJOIN_PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(RELJOIN_PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_pinned(urls: list[str], dest: Path, expected: str) -> None:
    if dest.exists() and sha256_file(dest) == expected:
        print("Already pinned:", dest.name)
        return
    last_err = None
    for url in urls:
        for attempt in range(4):
            try:
                print(f"GET {url} (try {attempt + 1})")
                urllib.request.urlretrieve(url, dest)
                got = sha256_file(dest)
                if got == expected:
                    print("OK", dest.name)
                    return
                last_err = RuntimeError(f"SHA mismatch for {dest.name}: {got}")
            except Exception as exc:
                last_err = exc
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Cannot pin {dest.name}: {last_err}")


fetch_pinned([HERBENCH_BASE_URL, JSDELIVR_BASE], EVAL_DIR / "base_vlm.py", HERBENCH_BASE_SHA256)
fetch_pinned([HERBENCH_ACC_URL, JSDELIVR_ACC], EVAL_DIR / "calculate_accuracy.py", HERBENCH_ACC_SHA256)

print({
    "RELJOIN_PERSIST": str(RELJOIN_PERSIST),
    "OUT_DIR": str(OUT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "EVAL_DIR": str(EVAL_DIR),
})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


CODE = REPO_DIR / "ovis"
link(CODE / "reljoin", ROOT / "reljoin")
for script in sorted(CODE.glob("run_*.py")):
    link(script, ROOT / script.name)
print("FrameWeave source linked from", CODE)


In [ ]:
# 4. Verify TSO abstention sources
from pathlib import Path

ROOT = Path("/content")
required = [
    ROOT / "run_tso_abstention.py",
    ROOT / "run_herbench_residual_direction.py",
    ROOT / "run_herbench_oracle_reljoin.py",
    ROOT / "reljoin" / "__init__.py",
    ROOT / "reljoin" / "config.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "ovis_hooks.py",
    ROOT / "reljoin" / "ovis_generate.py",
    ROOT / "reljoin" / "tso_abstention.py",
    ROOT / "reljoin" / "residual_direction.py",
    ROOT / "reljoin" / "herbench_oracle.py",
    ROOT / "reljoin" / "stats.py",
    ROOT / "reljoin" / "diagnostics.py",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("TSO abstention sources present.")
for p in required:
    print(f"  OK {p} ({p.stat().st_size} bytes)")


## 5. Gắn checkpoint (`outputs_reljoin_v6_ovis/checkpoints`)

Cùng layout train lite joint. Bắt buộc `Hybrid.pt` (`VARIANT=Hybrid`, `TRAIN_STAGE=joint`). Checkpoint v3/v4/v5 **không** load vào v6.0.

Ưu tiên `checkpoints/` trong repo đã clone. Drive chỉ dùng khi file đó không có.


In [ ]:
# Prefer the cloned checkpoint. Drive is the fallback in the next cell.
from __future__ import annotations

from pathlib import Path

ROOT = Path("/content")
REPO_DIR = Path("/content/FrameWeave")
PUBLISHED = REPO_DIR / "checkpoints" / "ovis"
OUT = ROOT / "outputs_reljoin_v6_ovis" / "checkpoints"

def link_published(src: Path, dest: Path) -> None:
    if not src.is_file():
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        return
    dest.symlink_to(src)

linked = []
for bucket in ("best_external_val", "last"):
    src = PUBLISHED / bucket / "Hybrid.pt"
    dest = OUT / bucket / "Hybrid.pt"
    link_published(src, dest)
    if dest.is_file():
        linked.append(str(dest))
print("published checkpoints", linked or ["none — next cell may use Drive"])


In [ ]:
# 5. Bind extract checkpoints — same layout as train.py / make_output_tree
from __future__ import annotations

import json
import shutil
import zipfile
from pathlib import Path

ROOT = Path("/content")
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / "outputs_tso_abstention").mkdir(parents=True, exist_ok=True)

try:
    REQUIRED_CKPTS
    EXTRACT_DIR_CANDIDATES
    VARIANT_TO_METHOD
except NameError:
    REQUIRED_CKPTS = ["Hybrid.pt"]
    VARIANT_TO_METHOD = {
        "Hybrid": "hybrid",
    }
    EXTRACT_DIR_CANDIDATES = [
        ROOT / "outputs_reljoin_v6_ovis",
        ROOT / "outputs_reljoin_v6_ovis (2)",
        Path("/content/drive/MyDrive/outputs_reljoin_v6_ovis"),
        Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs") / "outputs_reljoin_v6_ovis",
        Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs"),
    ]

v2_pt = ROOT / "outputs_reljoin_residual" / "checkpoints" / "best_external_val" / "RelJoin.pt"
if v2_pt.is_file():
    print("NOTE: found Ovis v2 RelJoin.pt at", v2_pt, "— không dùng cho v6.0")


def ckpt_complete(best_dir: Path) -> bool:
    return best_dir.is_dir() and all((best_dir / n).is_file() for n in REQUIRED_CKPTS)


def find_extract_root() -> Path:
    for cand in EXTRACT_DIR_CANDIDATES:
        best = cand / "checkpoints" / "best_external_val"
        if ckpt_complete(best):
            return cand
        last = cand / "checkpoints" / "last"
        if ckpt_complete(last):
            print("WARN: using checkpoints/last (best_external_val incomplete) at", cand)
            return cand
        if (cand / "Hybrid.pt").is_file():
            return cand
    zip_roots = [
        ROOT,
        Path("/content/drive/MyDrive"),
        Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs"),
    ]
    for base in zip_roots:
        if not base.exists():
            continue
        for zpath in list(base.glob("*reljoin*.zip")) + list(base.glob("*output*.zip")):
            if not zpath.is_file():
                continue
            dest = Path("/tmp/vast_outputs_reljoin_v6_ovis")
            if dest.exists():
                shutil.rmtree(dest)
            print("Extracting", zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(dest)
            for cand in [dest, *dest.iterdir()]:
                if cand.is_dir() and ckpt_complete(cand / "checkpoints" / "best_external_val"):
                    return cand
                if cand.is_dir() and ckpt_complete(cand / "checkpoints" / "last"):
                    print("WARN: zip extract using checkpoints/last at", cand)
                    return cand
    scanned = "\n".join(
        f"  {c} exists={c.exists()} best={list((c / 'checkpoints' / 'best_external_val').glob('*.pt')) if (c / 'checkpoints' / 'best_external_val').exists() else 'NO DIR'}"
        for c in EXTRACT_DIR_CANDIDATES
    )
    raise FileNotFoundError(
        "Không thấy Hybrid.pt đúng layout train:\n"
        "  {extract}/checkpoints/best_external_val/Hybrid.pt\n"
        "Upload cả folder `outputs_reljoin_v6_ovis` (kèm checkpoints/) lên /content hoặc Drive.\n"
        f"Đã quét:\n{scanned}"
    )


EXTRACT_DIR = find_extract_root()
BEST_DIR = EXTRACT_DIR / "checkpoints" / "best_external_val"
LAST_DIR = EXTRACT_DIR / "checkpoints" / "last"
if ckpt_complete(BEST_DIR):
    CKPT_DIR = BEST_DIR
elif ckpt_complete(LAST_DIR):
    CKPT_DIR = LAST_DIR
else:
    reljoin_loose = EXTRACT_DIR / "Hybrid.pt"
    CKPT_DIR = EXTRACT_DIR if reljoin_loose.is_file() else BEST_DIR
CKPT_DIR.mkdir(parents=True, exist_ok=True)

resolved = {}
for name in REQUIRED_CKPTS:
    src = CKPT_DIR / name
    if not src.is_file():
        alt = LAST_DIR / name
        if alt.is_file():
            print(f"best missing {name} → copy from last/")
            shutil.copy2(alt, CKPT_DIR / name)
            src = CKPT_DIR / name
        else:
            resolved[name] = None
            continue
    mb = src.stat().st_size / 1e6
    print(f"OK {name}: {src} ({mb:.1f} MB)")
    resolved[name] = src

missing = [n for n, p in resolved.items() if p is None]
assert not missing, f"Thiếu checkpoint: {missing} dưới {CKPT_DIR} và {LAST_DIR}"

if CKPT_DIR.resolve() != LAST_DIR.resolve():
    for variant in VARIANT_TO_METHOD:
        name = f"{variant}.pt"
        if not (CKPT_DIR / name).is_file() and (LAST_DIR / name).is_file():
            print(f"missing {name} → copy from last/")
            shutil.copy2(LAST_DIR / name, CKPT_DIR / name)

found_methods = ["vanilla"]
for variant, method in VARIANT_TO_METHOD.items():
    path = CKPT_DIR / f"{variant}.pt"
    if path.is_file():
        if path.name not in REQUIRED_CKPTS:
            mb = path.stat().st_size / 1e6
            print(f"OK {path.name}: {path} ({mb:.1f} MB)")
        found_methods.append(method)
METHODS = ",".join(found_methods)

for aux in ["train_summary.json", "config.json", "environment.txt"]:
    src = EXTRACT_DIR / aux
    if src.exists() and src.resolve() != (OUT_DIR / aux).resolve():
        shutil.copy2(src, OUT_DIR / aux)
        print("Copied", aux)
log_src = EXTRACT_DIR / "logs" / "train_metrics.csv"
if log_src.exists():
    (OUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
    dst = OUT_DIR / "logs" / "train_metrics.csv"
    if log_src.resolve() != dst.resolve():
        shutil.copy2(log_src, dst)

summary_path = EXTRACT_DIR / "train_summary.json"
if summary_path.exists():
    print("train_summary:", json.loads(summary_path.read_text()))

print({
    "EXTRACT_DIR": str(EXTRACT_DIR),
    "CKPT_DIR": str(CKPT_DIR),
    "METHODS": METHODS,
    "pts": sorted(p.name for p in CKPT_DIR.glob("*.pt")),
})


## 6. Tải Ovis2.5-9B + HERBench lite videos (~35 GB)

PerceptionTest **không** tải — không train. Cell này bắt buộc trước smoke (`model_config` phải `True`).


In [ ]:
# 6. Download Ovis2.5-9B + HERBench lite (eval only)
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_HF_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_HF_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"

for p in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent]:
    p.mkdir(parents=True, exist_ok=True)

print("Downloading Ovis2.5-9B (~18 GB bf16)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{MODEL_HF_ID}',
    revision='{MODEL_REVISION}',
    local_dir=r'{MODEL_DIR}',
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('model OK')
"""], check=True)
assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

print("Downloading HERBench lite (~35GB)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{HERBENCH_HF_ID}',
    repo_type='dataset',
    revision='{HERBENCH_REVISION}',
    local_dir=r'{HERBENCH_ROOT}',
    allow_patterns=[
        'data/herbench_lite_v2.parquet',
        'data/herbench_annotations_lite_v2.json',
        'data/*metadata.json',
        'videos/videos.tar.part.00',
        'videos/videos.tar.part.01',
        'videos/videos.tar.part.02',
        'videos/videos.tar.part.03',
        'videos/videos_lite_info.txt',
        'videos/videos.tar.checksums.txt',
    ],
)
print('herbench snapshot OK')
"""], check=True)

LITE_MP4_MIN = 68
parts = sorted(HERBENCH_VIDEOS.glob("videos.tar.part.*"))
print("parts:", [p.name for p in parts])
mp4s = list(HERBENCH_VIDEOS.rglob("*.mp4"))
assert len(parts) >= 4 or len(mp4s) >= 2, (
    f"Thiếu videos.tar.part.00–03 trong {HERBENCH_VIDEOS}. Có: {[p.name for p in parts]}"
)
if len(mp4s) < LITE_MP4_MIN:
    assert len(parts) >= 4, (
        f"Chỉ có {len(mp4s)} mp4 và thiếu tar parts — tải lại videos.tar.part.00–03."
    )
    cat = subprocess.Popen(["cat", *map(str, parts)], stdout=subprocess.PIPE)
    subprocess.run(["tar", "-xf", "-", "-C", str(HERBENCH_VIDEOS)], stdin=cat.stdout, check=True)
    cat.wait()
    print("Extracted archives")
else:
    print(f"Already have {len(mp4s)} mp4")

n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4")))
if n_mp4 >= LITE_MP4_MIN:
    for part in parts:
        part.unlink(missing_ok=True)
    print("Removed archives after full extract")
else:
    print(f"KEEP tar parts — only {n_mp4} mp4 (need ~{LITE_MP4_MIN} for full eval)")

print({
    "videos_mp4": n_mp4,
    "model_config": (MODEL_DIR / "config.json").exists(),
    "free_GiB": round(shutil.disk_usage(ROOT).free / 2**30, 2),
})
assert (MODEL_DIR / "config.json").exists(), "Ovis config.json missing — tải lại cell 6."
assert n_mp4 >= 2, "Lite extract xong vẫn thiếu mp4"
if n_mp4 < LITE_MP4_MIN:
    print(f"WARN: {n_mp4} mp4 < {LITE_MP4_MIN}. Smoke có thể chạy; full eval sẽ thiếu video.")
print("Downloads OK")


## 7. Smoke 8 câu rồi full 152

Một lần generate cho Vanilla (bypass) và một lần cho FrameWeave ($X+\Delta X$). Mask abstention chỉ sửa bản copy của score. Output nằm ở `outputs_tso_abstention/{smoke,full}`, không ghi đè `predictions_residual_direction`.

Eval chạy **process tách** (Stop cell không giết job). Log: `/content/tso_abstention_smoke.log` / `/content/tso_abstention_full.log`.

Nếu job cũ còn treo: `!pkill -f run_tso_abstention` rồi xóa `/content/tso_abstention.job.json`. Sau smoke OK đặt `SMOKE_FIRST = False`.


In [ ]:
# 7. TSO abstention — smoke 8 then full 152
from __future__ import annotations

import json
import os
import shlex
import shutil
import signal
import subprocess
import threading
import time
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
RUNNER = ROOT / "run_tso_abstention.py"
VENV = ROOT / "ovis_env"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"
TSO_ROOT = ROOT / "outputs_tso_abstention"
SMOKE_DIR = TSO_ROOT / "smoke"
FULL_DIR = TSO_ROOT / "full"
EVAL_DIR = ROOT / "herbench_eval_pins"
CKPT = CKPT_DIR if "CKPT_DIR" in dir() else (OUT_DIR / "checkpoints" / "best_external_val")
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
FULL_DIR.mkdir(parents=True, exist_ok=True)
JOB_FILE = ROOT / "tso_abstention.job.json"
DRIVE_PARENT = Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs")
DRIVE_DIR = DRIVE_PARENT / "tso_abstention"
if DRIVE_PARENT.is_dir():
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

def _from_cell1(name: str, default):
    """Read config from the Colab/IPython user namespace (cell 1), not globals()."""
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    if name in ns:
        return ns[name]
    return default

SMOKE_FIRST = bool(_from_cell1("SMOKE_FIRST", True))
RUN_FULL = bool(_from_cell1("RUN_FULL", True))
SMOKE_N = int(_from_cell1("SMOKE_N", 8))
RESULTS_PATH = Path(str(_from_cell1("RESULTS_PATH", "")))

HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"

needed = list(REQUIRED_CKPTS) if "REQUIRED_CKPTS" in dir() else ["Hybrid.pt"]
missing = [n for n in needed if not (CKPT / n).exists()]
assert not missing, (
    f"Chạy cell 5 trước. Thiếu {missing} trong {CKPT}. "
    "Upload folder `outputs_reljoin_v6_ovis/checkpoints/best_external_val/` lên /content."
)
assert (MODEL_DIR / "config.json").exists(), (
    f"Chạy cell 6 trước — missing {MODEL_DIR / 'config.json'} (model=false)."
)
assert PYTHON.exists(), "Chạy cell 2 trước — ovis_env missing."
assert RUNNER.is_file(), f"Missing {RUNNER}"

hy_pt = CKPT / "Hybrid.pt"
n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4"))) if HERBENCH_VIDEOS.exists() else 0
print({
    "Hybrid.pt": str(hy_pt),
    "Hybrid_MB": round(hy_pt.stat().st_size / 1e6, 1) if hy_pt.is_file() else None,
    "model_config": True,
    "MODEL_DIR": str(MODEL_DIR),
    "videos_mp4": n_mp4,
    "smoke_dir": str(SMOKE_DIR),
    "full_dir": str(FULL_DIR),
    "drive_dir": str(DRIVE_DIR) if DRIVE_DIR.exists() else None,
})
assert n_mp4 >= 2, f"Chạy cell 6 trước — chỉ có {n_mp4} mp4 trong {HERBENCH_VIDEOS}."

def resolve_results_path() -> Path:
    if str(RESULTS_PATH) and RESULTS_PATH.is_file():
        return RESULTS_PATH
    searches = [
        (ROOT / "outputs_reljoin_v6_ovis" / "predictions", "predictions_reljoin_*.csv"),
        (DRIVE_PARENT / "eval_full", "predictions_reljoin_*.csv"),
        (DRIVE_PARENT, "predictions_reljoin_*.csv"),
        (ROOT / "outputs_reljoin_v6_ovis" / "predictions_residual_direction", "predictions_residual_*.csv"),
        (DRIVE_PARENT / "residual_direction", "predictions_residual_*.csv"),
        (ROOT, "reljoin_v6_ovis_eval_*.zip"),
        (DRIVE_PARENT, "reljoin_v6_ovis_eval_*.zip"),
        (ROOT, "reljoin_v6_ovis_residual_direction_*.zip"),
        (DRIVE_PARENT, "reljoin_v6_ovis_residual_direction_*.zip"),
    ]
    for folder, pattern in searches:
        if not folder.is_dir():
            continue
        hits = [
            path for path in folder.glob(pattern)
            if path.is_file() and "smoke" not in path.parts
        ]
        uniq = []
        seen = set()
        for path in hits:
            key = str(path)
            if key in seen:
                continue
            seen.add(key)
            uniq.append(path)
        if len(uniq) == 1:
            return uniq[0]
        if len(uniq) > 1:
            raise FileNotFoundError(
                "Set RESULTS_PATH in cell 1 to one HERBench lite or residual-direction ZIP/CSV. "
                f"Found: {[str(path) for path in uniq]}"
            )
    raise FileNotFoundError(
        "Set RESULTS_PATH in cell 1 to the HERBench lite ZIP/CSV "
        "(predictions/predictions_reljoin_*.csv or reljoin_v6_ovis_eval_*.zip) "
        "or the residual-direction ZIP/CSV. Found: []"
    )


results_path = resolve_results_path()
shared_cmd = [
    str(PYTHON), "-u", str(RUNNER),
    "--results-path", str(results_path),
    "--checkpoint", str(CKPT / "Hybrid.pt"),
    "--root", "/content",
    "--herbench-root", str(HERBENCH_ROOT),
    "--videos-root", str(HERBENCH_VIDEOS),
    "--model-dir", str(MODEL_DIR),
    "--herbench-revision", HERBENCH_REVISION,
    "--model-revision", MODEL_REVISION,
    "--herbench-base-vlm", str(EVAL_DIR / "base_vlm.py"),
    "--herbench-base-sha256", HERBENCH_BASE_SHA256,
    "--max-pixels", "200704",
    "--min-pixels", "200704",
    "--num-frames", "8",
    "--max-new-tokens", "32",
    "--bootstrap", "20000",
    "--analysis-seed", "42",
]
smoke_cmd = shared_cmd + ["--output-dir", str(SMOKE_DIR), "--smoke", str(SMOKE_N)]
full_cmd = shared_cmd + ["--output-dir", str(FULL_DIR), "--smoke", "0", "--resume"]
print({"results": str(results_path), "smoke": SMOKE_N, "full_dir": str(FULL_DIR)})


def eval_env() -> dict:
    env = os.environ.copy()
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONNOUSERSITE"] = "1"
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    return env


def best_effort_copy(src: Path, dst: Path, timeout: float = 8.0) -> bool:
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return False
    try:
        if dst.exists() and src.resolve() == dst.resolve():
            return True
    except OSError:
        return False
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        print(f"WARN: mkdir {dst.parent}: {exc!r}", flush=True)
        return False
    err: list = []

    def _copy() -> None:
        try:
            tmp = dst.with_name(dst.name + ".tmp")
            shutil.copy2(src, tmp)
            tmp.replace(dst)
        except BaseException as exc:
            err.append(exc)

    t = threading.Thread(target=_copy, daemon=True)
    t.start()
    t.join(timeout)
    if t.is_alive():
        print(f"WARN: copy timed out {timeout}s: {src} -> {dst}", flush=True)
        return False
    if err:
        print(f"WARN: copy failed: {err[0]!r}", flush=True)
        return False
    return True


def _tso_artifacts(folder: Path) -> list[Path]:
    if not folder.is_dir():
        return []
    names = (
        "predictions.csv",
        "candidate_scores.jsonl",
        "tso_manifest.jsonl",
        "smoke_ids.json",
        "summary.json",
        "statistical_tests.json",
        "results.md",
        "results.tex",
        "config.json",
    )
    found = [folder / name for name in names if (folder / name).is_file()]
    found.extend(folder.glob("*.first_token_abandoned.*"))
    return found


def pull_drive_csvs() -> None:
    if not DRIVE_DIR.is_dir():
        return
    n = 0
    for src_root, dest_root in ((DRIVE_DIR / "smoke", SMOKE_DIR), (DRIVE_DIR / "full", FULL_DIR)):
        dest_root.mkdir(parents=True, exist_ok=True)
        for src in _tso_artifacts(src_root):
            if best_effort_copy(src, dest_root / src.name):
                n += 1
    if n:
        print(f"Pulled {n} artifact(s) from Drive -> {TSO_ROOT}", flush=True)


def push_drive_csvs() -> None:
    if not DRIVE_DIR.is_dir():
        return
    n = 0
    for src_root, dest_root in ((SMOKE_DIR, DRIVE_DIR / "smoke"), (FULL_DIR, DRIVE_DIR / "full")):
        if not src_root.is_dir():
            continue
        dest_root.mkdir(parents=True, exist_ok=True)
        for src in _tso_artifacts(src_root):
            if best_effort_copy(src, dest_root / src.name):
                n += 1
    if n:
        print(f"Pushed {n} artifact(s) -> {DRIVE_DIR}", flush=True)
    else:
        print(f"Drive push skipped or timed out — local files remain in {TSO_ROOT}", flush=True)


def _pid_alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


def _read_job() -> dict | None:
    if not JOB_FILE.exists():
        return None
    try:
        job = json.loads(JOB_FILE.read_text(encoding="utf-8"))
        pid = int(job["pid"])
    except Exception:
        return None
    if not _pid_alive(pid):
        JOB_FILE.unlink(missing_ok=True)
        return None
    return job


def _tail_log(log_path: Path, last_size: int) -> int:
    try:
        data = log_path.read_bytes()
    except OSError:
        return last_size
    if len(data) > last_size:
        print(data[last_size:].decode("utf-8", errors="replace"), end="", flush=True)
        return len(data)
    return last_size


def _kill_pg(pid: int) -> None:
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(pid, sig)
        except OSError:
            try:
                os.kill(pid, sig)
            except OSError:
                return
        if sig == signal.SIGTERM:
            time.sleep(2)


def _log_has_done(log_path: Path, label: str) -> bool:
    mark = "SMOKE OK" if label == "SMOKE" else "FULL OK"
    try:
        return mark in log_path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return False


def _wait_job(job: dict) -> bool:
    """True if worker exited. False if this cell was interrupted (worker still runs)."""
    pid = int(job["pid"])
    log_path = Path(job["log"])
    exit_path = Path(job["exit"])
    label = str(job.get("label") or "")
    last_size = 0
    silent_s = 0
    print(f"Waiting pid={pid} label={label} log={log_path}", flush=True)
    try:
        while True:
            prev = last_size
            last_size = _tail_log(log_path, last_size)
            if last_size == prev:
                silent_s += 15
                print(
                    f"[{time.strftime('%H:%M:%S')}] still running pid={pid} "
                    f"log={last_size}B silent={silent_s}s",
                    flush=True,
                )
            else:
                silent_s = 0
            if _log_has_done(log_path, label) and (not _pid_alive(pid) or silent_s >= 45):
                if _pid_alive(pid):
                    print(
                        f"{label} log already done; killing pid={pid} (CUDA exit hang).",
                        flush=True,
                    )
                    _kill_pg(pid)
                    time.sleep(1)
                if not exit_path.exists():
                    exit_path.write_text("0\n", encoding="utf-8")
                break
            if not _pid_alive(pid):
                time.sleep(1.0)
                last_size = _tail_log(log_path, last_size)
                break
            if silent_s >= 720:
                print(
                    "WARN: log silent >= 12 min (not a done marker). "
                    "Worker may be stuck; local CSV is still valid.",
                    flush=True,
                )
                silent_s = 0
            time.sleep(15)
    except KeyboardInterrupt:
        print(
            f"\nCell interrupted — {label} vẫn chạy pid={pid}. "
            f"Chạy lại cell này để reattach. Log: {log_path}",
            flush=True,
        )
        return False
    last_size = _tail_log(log_path, last_size)
    JOB_FILE.unlink(missing_ok=True)
    code = 1
    if exit_path.exists():
        try:
            code = int(exit_path.read_text(encoding="utf-8").strip() or "1")
        except ValueError:
            code = 1
    elif _log_has_done(log_path, label):
        code = 0
    if code != 0:
        raise RuntimeError(
            f"{label} exited {code}. "
            f"Xem {log_path}. "
            "venv torch / missing videos / state_dict / WARNING: no ckpt / missing Ovis config.json. "
            f"videos_mp4={n_mp4}. Câu thiếu mp4 được ghi runtime_error."
        )
    return True


def run_eval(cmd: list[str], label: str) -> bool:
    live = _read_job()
    if live is not None:
        print(
            f"Job đang chạy ({live.get('label')} pid={live['pid']}) — reattach, không start mới.",
            flush=True,
        )
        return _wait_job(live)

    log_path = ROOT / f"tso_abstention_{label.lower()}.log"
    exit_path = ROOT / f"tso_abstention_{label.lower()}.exit"
    exit_path.unlink(missing_ok=True)
    print("$", " ".join(cmd), flush=True)
    print("detached log:", log_path, flush=True)
    inner = " ".join(shlex.quote(c) for c in cmd)
    script = (
        f"{inner} > {shlex.quote(str(log_path))} 2>&1; "
        f"echo $? > {shlex.quote(str(exit_path))}"
    )
    proc = subprocess.Popen(
        ["bash", "-c", script],
        cwd="/content",
        env=eval_env(),
        start_new_session=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    job = {
        "pid": proc.pid,
        "label": label,
        "log": str(log_path),
        "exit": str(exit_path),
    }
    JOB_FILE.write_text(json.dumps(job), encoding="utf-8")
    print(f"started pid={proc.pid} (Stop cell không giết job)", flush=True)
    return _wait_job(job)


def _after_success(label: str) -> None:
    push_drive_csvs()
    if label == "SMOKE":
        print("SMOKE xong — kiểm tra log phải có 'Loaded FrameWeave v6 ckpt'.", flush=True)
    elif label == "FULL":
        print("FULL eval xong. Report trong", FULL_DIR, flush=True)


pull_drive_csvs()

live = _read_job()
if live is not None:
    ok = _wait_job(live)
    if not ok:
        print("Re-run cell 7 để reattach (đừng start job thứ hai).")
    else:
        _after_success(str(live.get("label") or ""))
        if live.get("label") == "SMOKE" and RUN_FULL:
            if n_mp4 < 68:
                print(f"WARN: {n_mp4} mp4. lite_v2 đủ thường là 68 file; câu TSO thiếu video sẽ lỗi.")
            if run_eval(list(full_cmd), "FULL"):
                _after_success("FULL")
            else:
                print("Full vẫn chạy nền. Re-run cell 7 để reattach.")
else:
    smoke_ok = True
    if SMOKE_FIRST:
        smoke_ok = run_eval(list(smoke_cmd), "SMOKE")
        if smoke_ok:
            _after_success("SMOKE")
        else:
            print("Smoke vẫn chạy nền. Re-run cell 7 để reattach (đừng start full song song).")
    if smoke_ok and RUN_FULL:
        if n_mp4 < 68:
            print(f"WARN: {n_mp4} mp4. lite_v2 đủ thường là 68 file; câu TSO thiếu video sẽ lỗi.")
        if run_eval(list(full_cmd), "FULL"):
            _after_success("FULL")
        else:
            print("Full vẫn chạy nền. Re-run cell 7 để reattach.")
    elif not RUN_FULL:
        print("RUN_FULL=False — chỉ smoke.")


## 8. Kết quả mask abstention

Đọc `outputs_tso_abstention/full/summary.json`. Run `incomplete` thì bảng không kết luận gain. Contrast chính là FrameWeave masked trừ Vanilla masked.


In [ ]:
# 8. Summarize TSO abstention mask
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

ROOT = Path("/content")
FULL = ROOT / "outputs_tso_abstention" / "full"
SMOKE = ROOT / "outputs_tso_abstention" / "smoke"


def show_dir(folder: Path) -> None:
    summary_path = folder / "summary.json"
    if not summary_path.is_file():
        print(f"Chưa có {summary_path}")
        return
    payload = json.loads(summary_path.read_text(encoding="utf-8"))
    print(folder.name, "fingerprint", payload.get("run_fingerprint"), "incomplete", payload.get("incomplete"))
    table = payload.get("table") or []
    if table:
        display(pd.DataFrame(table))
    if payload.get("incomplete"):
        print("Run incomplete. No gain is reported.", payload.get("incomplete_reasons"))
        return
    gains = payload.get("gains") or {}
    print(
        f"G_masked {gains.get('G_masked_pp')} pp, "
        f"difference of gains {gains.get('difference_pp')} pp"
    )
    stats_path = folder / "statistical_tests.json"
    if stats_path.is_file():
        stats = json.loads(stats_path.read_text(encoding="utf-8"))
        primary = stats.get("primary") or {}
        boot = stats.get("primary_bootstrap") or {}
        print(
            "primary",
            primary.get("name"),
            "p",
            (primary.get("mcnemar") or {}).get("p_value"),
            "CI pp",
            boot.get("ci_low_pp"),
            boot.get("ci_high_pp"),
            "videos",
            stats.get("n_videos"),
        )
    report = folder / "results.md"
    if report.is_file():
        print(report.read_text(encoding="utf-8"))


show_dir(SMOKE)
show_dir(FULL)


## 9. Copy artifact sang Drive


In [ ]:
# 9. Persist TSO abstention artifacts to Drive
from __future__ import annotations

import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/content")
TSO_ROOT = ROOT / "outputs_tso_abstention"
PERSIST = Path(os.environ.get("RELJOIN_PERSIST", "/content/drive/MyDrive/reljoin_v6_ovis_outputs"))
if not PERSIST.exists():
    print("Drive persist không có — bỏ qua copy.")
else:
    dest = PERSIST / "tso_abstention"
    try:
        same = dest.resolve() == TSO_ROOT.resolve()
    except OSError:
        same = False
    if same:
        print("Persist dest is the TSO output dir; skip copy.")
    else:
        dest.mkdir(parents=True, exist_ok=True)
        n = 0
        for name in ("smoke", "full"):
            src = TSO_ROOT / name
            if not src.is_dir():
                continue
            target = dest / name
            target.mkdir(parents=True, exist_ok=True)
            for path in src.iterdir():
                if path.is_file():
                    shutil.copy2(path, target / path.name)
                    n += 1
        (dest / "copied_at.txt").write_text(datetime.now(timezone.utc).isoformat(), encoding="utf-8")
        print(f"Copied {n} files → {dest}")


In [ ]:
# Zip TSO abstention results and download to your laptop
from __future__ import annotations

import zipfile
from datetime import datetime, timezone
from pathlib import Path

from google.colab import files

ROOT = Path("/content")
OUT = ROOT / "outputs_tso_abstention"
DRIVE = Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs") / "tso_abstention"

skip_dirs = {"oracle_clips", "checkpoints"}
skip_suffixes = {".pt", ".bin", ".safetensors"}

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"reljoin_v6_ovis_tso_abstention_{stamp}.zip"

n = 0
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    if OUT.exists():
        for path in OUT.rglob("*"):
            if not path.is_file():
                continue
            if any(part in skip_dirs for part in path.relative_to(OUT).parts):
                continue
            if path.suffix.lower() in skip_suffixes:
                continue
            zf.write(path, arcname=f"outputs_tso_abstention/{path.relative_to(OUT)}")
            n += 1
    if DRIVE.exists():
        for path in DRIVE.rglob("*"):
            if not path.is_file():
                continue
            if any(part in skip_dirs for part in path.relative_to(DRIVE).parts):
                continue
            if path.suffix.lower() in skip_suffixes:
                continue
            zf.write(path, arcname=f"drive_tso_abstention/{path.relative_to(DRIVE)}")
            n += 1

print(f"Packed {n} files → {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
files.download(str(zip_path))
